In [ ]:
import pandas as pd

BASE_PATH = "/content/drive/MyDrive/NLP"

csv_files = {
    "phishing_email.csv": f"{BASE_PATH}/phishing_email.csv",
    "Nigerian_Fraud.csv": f"{BASE_PATH}/Nigerian_Fraud.csv",
    "Ling.csv": f"{BASE_PATH}/Ling.csv",
    "CEAS_08.csv": f"{BASE_PATH}/CEAS_08.csv",
    "SpamAssasin.csv": f"{BASE_PATH}/SpamAssasin.csv",
    "Nazario.csv": f"{BASE_PATH}/Nazario.csv",
    "Enron.csv": f"{BASE_PATH}/Enron.csv"
}

datasets = {}

for name, path in csv_files.items():
    print(f"\nLoading {name}...")

    df = pd.read_csv(
        path,
        encoding='latin1',
        engine='python',        # FIX 1: use python engine
        on_bad_lines='skip'     # FIX 2: skip broken lines
    )

    datasets[name] = df
    print(f"Loaded {name} → shape = {df.shape}")

print("\nAll CSV files loaded successfully!")



Loading phishing_email.csv...
Loaded phishing_email.csv → shape = (34252, 2)

Loading Nigerian_Fraud.csv...
Loaded Nigerian_Fraud.csv → shape = (3332, 7)

Loading Ling.csv...
Loaded Ling.csv → shape = (2859, 3)

Loading CEAS_08.csv...
Loaded CEAS_08.csv → shape = (16675, 7)

Loading SpamAssasin.csv...
Loaded SpamAssasin.csv → shape = (7863, 7)

Loading Nazario.csv...
Loaded Nazario.csv → shape = (1563, 7)

Loading Enron.csv...
Loaded Enron.csv → shape = (29037, 3)

All CSV files loaded successfully!


In [ ]:
def get_label(df, dataset_name):
    name = dataset_name.lower()

    # USE TRUE LABELS ONLY FOR phishing_email.csv
    if "phishing_email" in name:

        # make a copy so we don't modify original df
        lbl = df["label"].copy()

        # Fill missing values with 0 (safe fallback)
        lbl = lbl.fillna(0)

        # Convert safely to int
        return lbl.astype(int)

    # PHISHING datasets (no label column or corrupted)
    if "nigerian" in name or "fraud" in name or "nazario" in name or "ceas" in name:
        return pd.Series([1] * len(df))

    # LEGITIMATE datasets
    if "spamassasin" in name or "enron" in name or "ling" in name:
        return pd.Series([0] * len(df))

    # fallback
    return pd.Series([0] * len(df))


In [ ]:
def extract_email_text(df):
    if "subject" in df.columns and "body" in df.columns:
        return (df["subject"].fillna("") + " " + df["body"].fillna("")).astype(str)

    if "text_combined" in df.columns:
        return df["text_combined"].astype(str)

    for col in ["body", "text", "message", "content", "email", "email_text"]:
        if col in df.columns:
            return df[col].fillna("").astype(str)

    text_cols = df.select_dtypes(include=['object']).columns
    return df[text_cols].astype(str).agg(" ".join, axis=1)


In [ ]:
merged_rows = []

for name, df in datasets.items():
    temp = pd.DataFrame()
    temp["email_text"] = extract_email_text(df)
    temp["label"] = get_label(df, name)
    temp["source"] = name
    merged_rows.append(temp)

df_all = pd.concat(merged_rows, ignore_index=True)
df_all["label"].value_counts()


,count
label,
0,58617
1,36964


# URL Extraction Code

In [ ]:
import re

# URL regex pattern (works for http, https, www)
url_pattern = re.compile(r'(https?://\S+|www\.\S+)', re.IGNORECASE)

def extract_urls(text):
    if not isinstance(text, str):
        return []
    return re.findall(url_pattern, text)

df_all["urls"] = df_all["email_text"].apply(extract_urls)

# Extract first URL for feature engineering
df_all["first_url"] = df_all["urls"].apply(lambda x: x[0] if len(x) > 0 else "")

df_all[["email_text", "first_url"]].head()


,email_text,first_url
0,hpl nom may 25 2001 see attached file hplno 52...,
1,nom actual vols 24 th forwarded sabrae zajac h...,
2,enron actuals march 30 april 1 201 estimated a...,
3,hpl nom may 30 2001 see attached file hplno 53...,
4,hpl nom june 1 2001 see attached file hplno 60...,


# URL FEATURE ENGINEERING

In [ ]:
!pip install tldextract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 2.7 MB/s eta 0:00:00


In [ ]:
import re

# URL regex pattern
url_pattern = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)

def extract_urls(text):
    if not isinstance(text, str):
        return []
    return re.findall(url_pattern, text)

# Extract URL list
df_all["urls"] = df_all["email_text"].apply(extract_urls)

# Extract first URL
df_all["first_url"] = df_all["urls"].apply(lambda x: x[0] if len(x) > 0 else "")

df_all[["email_text", "first_url"]].head()


,email_text,first_url
0,hpl nom may 25 2001 see attached file hplno 52...,
1,nom actual vols 24 th forwarded sabrae zajac h...,
2,enron actuals march 30 april 1 201 estimated a...,
3,hpl nom may 30 2001 see attached file hplno 53...,
4,hpl nom june 1 2001 see attached file hplno 60...,


# NEW CELL 1 — Rename Wrong Function (SAFE FIX)

In [ ]:
# Fix: rename the mistakenly overwritten function
def extract_urls_only(text):
    if not isinstance(text, str):
        return []
    return re.findall(url_pattern, text)


# NEW CELL 2 — Correct URL FEATURE ENGINEERING Function

In [ ]:
import math
from urllib.parse import urlparse
import tldextract

def calc_entropy(s):
    if not s:
        return 0
    prob = [float(s.count(c)) / len(s) for c in dict.fromkeys(list(s))]
    entropy = - sum([p * math.log(p, 2) for p in prob])
    return entropy

def contains_ip(url):
    ip_pattern = r'\b(?:\d{1,3}\.){3}\d{1,3}\b'
    return 1 if re.search(ip_pattern, url) else 0

def extract_url_features(url):
    """ RETURNS 8 URL FEATURES — DO NOT MODIFY THIS FUNCTION """
    if not url or not isinstance(url, str):
        return pd.Series([0, 0, 0, 0, 0, "", "", 0])

    try:
        parsed = urlparse(url)
    except:
        return pd.Series([0, 0, 0, 0, 0, "", "", 0])

    extracted = tldextract.extract(url)

    length = len(url)
    num_dots = url.count(".")
    special_chars = sum(not c.isalnum() for c in url)
    https_flag = 1 if url.lower().startswith("https") else 0
    ip_flag = contains_ip(url)
    entropy = calc_entropy(url)

    domain = extracted.domain if extracted.domain else ""
    tld = extracted.suffix if extracted.suffix else ""

    return pd.Series([
        length, num_dots, special_chars, https_flag,
        ip_flag, domain, tld, entropy
    ])


In [ ]:
df_all[[
    "url_length", "url_dots", "url_special_chars", "url_https",
    "url_ip_flag", "url_domain", "url_tld", "url_entropy"
]] = df_all["first_url"].apply(extract_url_features)


In [ ]:
datasets["phishing_email.csv"].head()
datasets["phishing_email.csv"].columns
datasets["phishing_email.csv"]["label"].value_counts()


,count
label,
0.0,18857
1.0,15394


In [ ]:
df_all[["first_url", "url_length", "url_entropy"]].head()


,first_url,url_length,url_entropy
0,,0,0.0
1,,0,0.0
2,,0,0.0
3,,0,0.0
4,,0,0.0


#

In [ ]:
import re

ip_pattern = re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}\b")

def extract_ip(text):
    if not isinstance(text, str):
        return ""
    matches = ip_pattern.findall(text)
    return matches[0] if matches else ""

df_all["sender_ip"] = df_all["email_text"].apply(extract_ip)
df_all["sender_ip"].head(10)


,sender_ip
0,
1,
2,
3,
4,
5,
6,
7,
8,
9,


In [ ]:
import re

ip_pattern = re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}\b")

def extract_ip_from_url(url):
    matches = ip_pattern.findall(url)
    return matches[0] if matches else ""

df_all["ip_from_url"] = df_all["first_url"].apply(extract_ip_from_url)

df_all[["first_url", "ip_from_url"]].head(10)


,first_url,ip_from_url
0,,
1,,
2,,
3,,
4,,
5,,
6,,
7,,
8,,
9,,


In [ ]:
df_with_urls = df_all[df_all["first_url"] != ""]
df_with_urls[["first_url", "ip_from_url"]].head(20)


,first_url,ip_from_url
34258,http://login.mail.lycos.com/brandPage.shtml?pa...,
34261,http://login.mail.lycos.com/brandPage.shtml?pa...,
34267,www.swissmoney.net/briefs.htm,
34271,http://webhosting.yahoo.com,
34275,http://phantomemail.com/.,
34279,http://Game.37.com/,
34283,http://Game.37.com/,
34292,http://Game.37.com/,
34293,http://mail.go.com,
34295,http://www.zwallet.com/index.html?user=ga_mm1,


In [ ]:
df_phish_urls = df_all[(df_all["source"].isin(["Nigerian_Fraud.csv", "Nazario.csv", "CEAS_08.csv"])) & (df_all["first_url"] != "")]
df_phish_urls[["source", "first_url", "ip_from_url"]].head(20)


,source,first_url,ip_from_url
34258,Nigerian_Fraud.csv,http://login.mail.lycos.com/brandPage.shtml?pa...,
34261,Nigerian_Fraud.csv,http://login.mail.lycos.com/brandPage.shtml?pa...,
34267,Nigerian_Fraud.csv,www.swissmoney.net/briefs.htm,
34271,Nigerian_Fraud.csv,http://webhosting.yahoo.com,
34275,Nigerian_Fraud.csv,http://phantomemail.com/.,
34279,Nigerian_Fraud.csv,http://Game.37.com/,
34283,Nigerian_Fraud.csv,http://Game.37.com/,
34292,Nigerian_Fraud.csv,http://Game.37.com/,
34293,Nigerian_Fraud.csv,http://mail.go.com,
34295,Nigerian_Fraud.csv,http://www.zwallet.com/index.html?user=ga_mm1,


# Add IP blacklist features even if ip_from_url is empty

In [ ]:
import pandas as pd

# FireHOL Level 1 blacklist (most widely used)
firehol_url = "https://raw.githubusercontent.com/firehol/blocklist-ipsets/master/firehol_level1.netset"
firehol = pd.read_csv(firehol_url, comment="#", header=None, names=["ip"])
firehol["source"] = "firehol"

# Spamhaus DROP list
spamhaus_url = "https://www.spamhaus.org/drop/drop.txt"
spamhaus = pd.read_csv(spamhaus_url, comment=";", header=None, sep=" ", names=["ip", "comment"])
spamhaus = spamhaus[["ip"]]
spamhaus["source"] = "spamhaus"

# Blocklist.de Attacker IPs
blocklist_url = "https://lists.blocklist.de/lists/all.txt"
blocklist = pd.read_csv(blocklist_url, header=None, names=["ip"])
blocklist["source"] = "blocklist"

# Combine all
ip_blacklist = pd.concat([firehol, spamhaus, blocklist], ignore_index=True)

len(ip_blacklist)


29432

In [ ]:
# Remove invalid rows (non-IP strings)
ip_blacklist["ip"] = ip_blacklist["ip"].astype(str).str.strip()
ip_blacklist = ip_blacklist[ip_blacklist["ip"].str.match(r"\d{1,3}(?:\.\d{1,3}){3}")]
ip_blacklist = ip_blacklist.drop_duplicates().reset_index(drop=True)

ip_blacklist.shape


(29369, 2)

In [ ]:
bad_ip_set = set(ip_blacklist["ip"])
len(bad_ip_set)


27985

In [ ]:
df_all["ip_blacklisted"] = df_all["ip_from_url"].apply(
    lambda ip: 1 if ip in bad_ip_set else 0
)

df_all["ip_blacklisted"].value_counts()


,count
ip_blacklisted,
0,95581


# TEXT FEATURE ENGINEERING (TF-IDF VECTORISATION)

In [ ]:
import re
import string

def clean_text(text):
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)  # remove numbers
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_all["clean_text"] = df_all["email_text"].apply(clean_text)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X_text = tfidf.fit_transform(df_all["clean_text"])
X_text.shape


(95581, 5000)

In [ ]:
import numpy as np

url_ip_features = df_all[[
    "url_length",
    "url_dots",
    "url_special_chars",
    "url_https",
    "url_ip_flag",
    "url_entropy",
    "ip_blacklisted"
]].fillna(0).astype(float)

X_other = url_ip_features.values
X_other.shape


(95581, 7)

In [ ]:
from scipy.sparse import hstack

X = hstack([X_text, X_other])
X.shape


(95581, 5007)

In [ ]:
y = df_all["label"].values
y.shape


(95581,)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'
)

model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.8724695297379296
              precision    recall  f1-score   support

           0       0.89      0.90      0.90     11724
           1       0.84      0.82      0.83      7393

    accuracy                           0.87     19117
   macro avg       0.87      0.86      0.86     19117
weighted avg       0.87      0.87      0.87     19117



# MODEL-2 → URL-Only Classifier (using URL.csv dataset)

In [ ]:
url_df = pd.read_csv("/content/drive/MyDrive/NLP/URL.csv", encoding='latin1')
url_df.head()
url_df.columns
url_df.shape
url_df['label'].value_counts()  # if label column exists


,count
label,
benign,345738
malicious,104438


In [ ]:
url_path = "/content/drive/MyDrive/NLP/URL.csv"  # adjust if file name differs

url_df = pd.read_csv(url_path, encoding='latin1')

print(url_df.shape)
print(url_df.columns)
url_df.head()


(450176, 4)
Index(['Unnamed: 0', 'url', 'label', 'result'], dtype='object')


,Unnamed: 0,url,label,result
0,0,https://www.google.com,benign,0
1,1,https://www.youtube.com,benign,0
2,2,https://www.facebook.com,benign,0
3,3,https://www.baidu.com,benign,0
4,4,https://www.wikipedia.org,benign,0


In [ ]:
# Drop unwanted column
url_df = url_df.drop(columns=["Unnamed: 0"], errors='ignore')

# Standardize URL column
url_df["url"] = url_df["url"].astype(str).str.strip().str.lower()

# Convert labels → numeric
url_df["label"] = url_df["label"].str.lower().map({
    "benign": 0,
    "malicious": 1
})


In [ ]:
url_df[[
    "url_length","url_dots","url_special_chars","url_https",
    "url_ip_flag","url_entropy"
]] = url_df["url"].apply(
    lambda u: pd.Series([
        len(u),
        u.count("."),
        sum(not c.isalnum() for c in u),
        1 if u.startswith("https") else 0,
        contains_ip(u),
        calc_entropy(u)
    ])
)


In [ ]:
import tldextract

def extract_domain_tld(u):
    ext = tldextract.extract(u)
    return pd.Series([ext.domain or "", ext.suffix or ""])

url_df[["domain", "tld"]] = url_df["url"].apply(extract_domain_tld)


In [ ]:
from sklearn.preprocessing import LabelEncoder

domain_le = LabelEncoder()
tld_le = LabelEncoder()

url_df["domain_enc"] = domain_le.fit_transform(url_df["domain"])
url_df["tld_enc"] = tld_le.fit_transform(url_df["tld"])


In [ ]:
X_url = url_df[[
    "url_length", "url_dots", "url_special_chars", "url_https",
    "url_ip_flag", "url_entropy", "domain_enc", "tld_enc"
]].values

y_url = url_df["label"].values

X_url.shape, y_url.shape


((450176, 8), (450176,))

In [ ]:
from sklearn.model_selection import train_test_split

X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(
    X_url, y_url, test_size=0.2, random_state=42, stratify=y_url
)


In [ ]:
from xgboost import XGBClassifier

url_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'  # MUCH faster
)

url_model.fit(X_train_u, y_train_u)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_u = url_model.predict(X_test_u)

print("URL Model Accuracy:", accuracy_score(y_test_u, y_pred_u))
print(classification_report(y_test_u, y_pred_u))


URL Model Accuracy: 0.9935470256341907
              precision    recall  f1-score   support

           0       0.99      1.00      1.00     69148
           1       1.00      0.98      0.99     20888

    accuracy                           0.99     90036
   macro avg       0.99      0.99      0.99     90036
weighted avg       0.99      0.99      0.99     90036



# MODEL-3: IP Reputation Model (Rule-Based, Industry Standard)

In [ ]:
# Extract IPv4 addresses from URL
import re

def extract_ip(url):
    if not isinstance(url, str):
        return ""
    match = re.search(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', url)
    return match.group(0) if match else ""

df_all["ip_from_url"] = df_all["first_url"].apply(extract_ip)
df_all["ip_from_url"].head(20)


,ip_from_url
0,
1,
2,
3,
4,
5,
6,
7,
8,
9,


In [ ]:
# Convert blacklisted IPs into a fast lookup set
bad_ip_set = set(ip_blacklist["ip"])
len(bad_ip_set)


27985

In [ ]:
def ip_reputation(ip):
    if isinstance(ip, str) and ip in bad_ip_set:
        return 1   # HIGH RISK
    return 0       # SAFE


In [ ]:
df_all["ip_risk"] = df_all["ip_from_url"].apply(ip_reputation)
df_all["ip_risk"].value_counts()


,count
ip_risk,
0,95581


In [ ]:
ip_reputation("185.22.11.1")


0

In [ ]:
ip_reputation("142.250.74.14")


0

In [ ]:
"185.22.11.1" in bad_ip_set


False

In [ ]:
import requests

extra_urls = [
    "https://rules.emergingthreats.net/blockrules/compromised-ips.txt",
    "https://rules.emergingthreats.net/fwrules/emerging-Block-IPs.txt",
]

extra_ip_list = []

for url in extra_urls:
    try:
        print("Loading:", url)
        text = requests.get(url).text
        lines = text.split("\n")

        # Keep only valid IPv4 entries
        for line in lines:
            line = line.strip()
            if line and line.count(".") == 3 and "/" not in line:  # remove CIDR
                extra_ip_list.append(line)

        print(f"Loaded {len(extra_ip_list)} IPs so far.")
    except Exception as e:
        print("Error:", e)

len(extra_ip_list)


Loading: https://rules.emergingthreats.net/blockrules/compromised-ips.txt
Loaded 467 IPs so far.
Loading: https://rules.emergingthreats.net/fwrules/emerging-Block-IPs.txt
Loaded 471 IPs so far.


471

In [ ]:
combined_ips = set(ip_blacklist["ip"]).union(extra_ip_list)

print("Old blacklist size:", len(ip_blacklist))
print("New blacklist size:", len(combined_ips))


Old blacklist size: 29369
New blacklist size: 28389


# MULTIMODAL FUSION MODEL

In [ ]:
"text_model" in globals()


False

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X_text = tfidf.fit_transform(df_all["clean_text"])
print("TF-IDF shape:", X_text.shape)


TF-IDF shape: (95581, 5000)


In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X_text, df_all["label"], test_size=0.2, random_state=42
)

text_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'
)

text_model.fit(X_train, y_train)

print("Model-1 training complete!")


Model-1 training complete!


In [ ]:
"text_model" in globals()


True

In [ ]:
import joblib
joblib.dump(text_model, "/content/text_model.pkl")
joblib.dump(tfidf, "/content/tfidf.pkl")


['/content/tfidf.pkl']

## Later you can load instantly

In [ ]:
text_model = joblib.load("/content/text_model.pkl")
tfidf = joblib.load("/content/tfidf.pkl")


# m2

In [ ]:
from sklearn.preprocessing import LabelEncoder

domain_encoder = LabelEncoder()
tld_encoder = LabelEncoder()

df_all["url_domain_enc"] = domain_encoder.fit_transform(df_all["url_domain"].astype(str))
df_all["url_tld_enc"] = tld_encoder.fit_transform(df_all["url_tld"].astype(str))


In [ ]:
url_feature_columns = [
    "url_length",
    "url_dots",
    "url_special_chars",
    "url_https",
    "url_ip_flag",
    "url_entropy",
    "url_domain_enc",
    "url_tld_enc"
]


In [ ]:
df_all["url_prob"] = url_model.predict_proba(df_all[url_feature_columns])[:, 1]


In [ ]:
# Model-1 probability
df_all["text_prob"] = text_model.predict_proba(X_text)[:, 1]

# Model-2 probability
df_all["url_prob"] = url_model.predict_proba(df_all[url_feature_columns])[:, 1]

# Model-3
df_all["ip_risk"] = df_all["ip_from_url"].apply(ip_reputation)

df_all[["text_prob", "url_prob", "ip_risk"]].head()


,text_prob,url_prob,ip_risk
0,0.022521,0.999993,0
1,0.000104,0.999993,0
2,0.006554,0.999993,0
3,0.022521,0.999993,0
4,0.018147,0.999993,0


In [ ]:
fusion_features = [
    "text_prob",
    "url_prob",
    "ip_risk",
    "url_length",
    "url_dots",
    "url_special_chars",
    "url_https",
    "url_ip_flag",
    "url_entropy"
]

X_fusion = df_all[fusion_features]
y_fusion = df_all["label"]

X_fusion.head()


,text_prob,url_prob,ip_risk,url_length,url_dots,url_special_chars,url_https,url_ip_flag,url_entropy
0,0.022521,0.999993,0,0,0,0,0,0,0.0
1,0.000104,0.999993,0,0,0,0,0,0,0.0
2,0.006554,0.999993,0,0,0,0,0,0,0.0
3,0.022521,0.999993,0,0,0,0,0,0,0.0
4,0.018147,0.999993,0,0,0,0,0,0,0.0


In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fusion, y_fusion, test_size=0.2, random_state=42
)

fusion_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'
)

fusion_model.fit(X_train_f, y_train_f)

print("Fusion Model trained!")


Fusion Model trained!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_f = fusion_model.predict(X_test_f)

print("Fusion Model Accuracy:", accuracy_score(y_test_f, y_pred_f))
print(classification_report(y_test_f, y_pred_f))


Fusion Model Accuracy: 0.8861746089867657
              precision    recall  f1-score   support

           0       0.94      0.87      0.90     11753
           1       0.82      0.91      0.86      7364

    accuracy                           0.89     19117
   macro avg       0.88      0.89      0.88     19117
weighted avg       0.89      0.89      0.89     19117



In [ ]:
import joblib

# Save TF-IDF vectorizer
joblib.dump(tfidf, "/content/tfidf.pkl")

# Save Model-1: Text classifier
joblib.dump(text_model, "/content/text_model.pkl")

# Save Model-2: URL classifier
joblib.dump(url_model, "/content/url_model.pkl")

# Save Model-3: IP reputation (bad IP set)
joblib.dump(bad_ip_set, "/content/ip_blacklist.pkl")

# Save Model-4: Final multimodal fusion model
joblib.dump(fusion_model, "/content/fusion_model.pkl")

print("All models saved successfully!")


All models saved successfully!


In [ ]:
%%writefile /content/drive/MyDrive/NLP/app/app.py
import streamlit as st
import re
import joblib
import tldextract
import math
import numpy as np
from urllib.parse import urlparse

BASE = "/content/drive/MyDrive/NLP/app/"

tfidf = joblib.load(BASE + "tfidf.pkl")
text_model = joblib.load(BASE + "text_model.pkl")
url_model = joblib.load(BASE + "url_model.pkl")
fusion_model = joblib.load(BASE + "fusion_model.pkl")
bad_ip_set = joblib.load(BASE + "ip_blacklist.pkl")
domain_encoder = joblib.load(BASE + "domain_encoder.pkl")
tld_encoder = joblib.load(BASE + "tld_encoder.pkl")

url_pattern = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
ip_pattern = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")

def extract_urls(text):
    return re.findall(url_pattern, text) if isinstance(text, str) else []

def extract_ip(url):
    match = re.search(ip_pattern, url)
    return match.group(0) if match else ""

def clean_text(text):
    return re.sub(r"[^a-zA-Z0-9 ]", " ", str(text).lower())

def calc_entropy(s):
    if not s:
        return 0
    from collections import Counter
    probs = [v/len(s) for v in Counter(s).values()]
    return -sum(p * math.log2(p) for p in probs)

def extract_url_features(url):
    if not url:
        return [0]*8
    ext = tldextract.extract(url)
    length = len(url)
    dots = url.count(".")
    special = sum(not c.isalnum() for c in url)
    https = 1 if url.startswith("https") else 0
    ip_flag = 1 if extract_ip(url) else 0
    entropy = calc_entropy(url)
    domain = ext.domain or ""
    tld = ext.suffix or ""
    domain_enc = domain_encoder.transform([domain])[0] if domain in domain_encoder.classes_ else 0
    tld_enc = tld_encoder.transform([tld])[0] if tld in tld_encoder.classes_ else 0
    return [length, dots, special, https, ip_flag, entropy, domain_enc, tld_enc]

def ip_reputation(ip):
    return 1 if ip in bad_ip_set else 0

st.title("🔐 Multimodal Phishing Detection System")
st.write("Text + URL + IP based Phishing Detection")

email_text = st.text_area("Paste Email Content Here", height=250)

if st.button("Analyze Email"):
    urls = extract_urls(email_text)
    first_url = urls[0] if urls else ""
    url_features = extract_url_features(first_url)
    url_prob = url_model.predict_proba([url_features])[0][1] if first_url else 0
    text_vec = tfidf.transform([clean_text(email_text)])
    text_prob = text_model.predict_proba(text_vec)[0][1]
    ip_addr = extract_ip(first_url)
    ip_risk = ip_reputation(ip_addr)
    fusion_input = np.array([[text_prob, url_prob, ip_risk] + url_features[:6]])
    final_pred = fusion_model.predict(fusion_input)[0]
    final_prob = fusion_model.predict_proba(fusion_input)[0][1]

    st.subheader("Result")
    st.success("LEGITIMATE" if final_pred == 0 else "PHISHING")
    st.write(f"Confidence: {final_prob:.4f}")
    st.write(f"Text Risk: {text_prob:.4f}")
    st.write(f"URL Risk: {url_prob:.4f}")
    st.write(f"IP Risk: {ip_risk}")


Writing /content/drive/MyDrive/NLP/app/app.py


In [ ]:
!sed -n '1,40p' /content/drive/MyDrive/NLP/app/app.py


import streamlit as st
import re
import joblib
import tldextract
import math
import numpy as np
from urllib.parse import urlparse

BASE = "/content/drive/MyDrive/NLP/app/"

tfidf = joblib.load(BASE + "tfidf.pkl")
text_model = joblib.load(BASE + "text_model.pkl")
url_model = joblib.load(BASE + "url_model.pkl")
fusion_model = joblib.load(BASE + "fusion_model.pkl")
bad_ip_set = joblib.load(BASE + "ip_blacklist.pkl")
domain_encoder = joblib.load(BASE + "domain_encoder.pkl")
tld_encoder = joblib.load(BASE + "tld_encoder.pkl")

url_pattern = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
ip_pattern = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")

def extract_urls(text):
    return re.findall(url_pattern, text) if isinstance(text, str) else []

def extract_ip(url):
    match = re.search(ip_pattern, url)
    return match.group(0) if match else ""

def clean_text(text):
    return re.sub(r"[^a-zA-Z0-9 ]", " ", str(text).lower())

def calc_entropy(s):
    if not s:
        return 0
    from 

In [ ]:
!pkill -f streamlit
!pkill -f ngrok
!pkill -f cloudflared


In [ ]:
!streamlit run /content/drive/MyDrive/NLP/app/app.py





  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.196.102.15:8501

  Stopping...
  Stopping...


In [ ]:
!pkill -f streamlit
